## Import data

In [10]:

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import imageio.v2 as imageio
import cv2
import os

torch.cuda.empty_cache()
#torch.cuda.init()
#torch.cuda.reset_peak_memory_stats()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Running on: ",device)

class LoadFrames(Dataset):
    def __init__(self, video_path):
        reader = imageio.get_reader(video_path)
        self.frames = [cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)/255.0 for frame in reader]
        reader.close()

    def __len__(self):
        return len(self.frames) - 1

    def __getitem__(self, idx):
        input_frame = self.frames[idx][None, :, :]
        target_frame = self.frames[idx + 1][None, :, :]
        return torch.tensor(input_frame, dtype=torch.float32), torch.tensor(target_frame, dtype=torch.float32)

video_dataset = LoadFrames("test.mp4")
loader = DataLoader(video_dataset, batch_size=16, shuffle=True)

Running on:  cuda


## Define model. In this case we are going to use a U-Net

In [19]:

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super(UNet, self).__init__()
        self.enc1 = DoubleConv(in_channels, 32)
        self.pool = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)
        self.out_conv = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool(enc1))
        enc3 = self.enc3(self.pool(enc2))
        dec2 = self.dec2(torch.cat([self.up2(enc3), enc2], dim=1))
        dec1 = self.dec1(torch.cat([self.up1(dec2), enc1], dim=1))
        return torch.sigmoid(self.out_conv(dec1))

model = UNet().to(device)
criterion = nn.BCELoss() 
optimizer = optim.Adam(model.parameters(), lr=0.1)

epochs = 40
for epoch in range(epochs):
    running_loss = 0.0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        inputs += 0.01 * torch.randn_like(inputs)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(loader):.5f}")

# Save model
torch.save(model.state_dict(), "unet_frame_predictor.pth")
print("Model saved to unet_frame_predictor.pth")


Epoch 1/40, Loss: 0.12010
Epoch 2/40, Loss: 0.06949
Epoch 3/40, Loss: 0.05484
Epoch 4/40, Loss: 0.04693
Epoch 5/40, Loss: 0.04418
Epoch 6/40, Loss: 0.04036
Epoch 7/40, Loss: 0.04146
Epoch 8/40, Loss: 0.03754
Epoch 9/40, Loss: 0.03485
Epoch 10/40, Loss: 0.03520
Epoch 11/40, Loss: 0.03308
Epoch 12/40, Loss: 0.03105
Epoch 13/40, Loss: 0.03075
Epoch 14/40, Loss: 0.03151
Epoch 15/40, Loss: 0.02865
Epoch 16/40, Loss: 0.03028
Epoch 17/40, Loss: 0.03076
Epoch 18/40, Loss: 0.02851
Epoch 19/40, Loss: 0.02804
Epoch 20/40, Loss: 0.02825
Epoch 21/40, Loss: 0.02764
Epoch 22/40, Loss: 0.02748
Epoch 23/40, Loss: 0.02661
Epoch 24/40, Loss: 0.02619
Epoch 25/40, Loss: 0.02633
Epoch 26/40, Loss: 0.02636
Epoch 27/40, Loss: 0.02692
Epoch 28/40, Loss: 0.02625
Epoch 29/40, Loss: 0.02781
Epoch 30/40, Loss: 0.02683
Epoch 31/40, Loss: 0.02625
Epoch 32/40, Loss: 0.02721
Epoch 33/40, Loss: 0.02874
Epoch 34/40, Loss: 0.02737
Epoch 35/40, Loss: 0.02816
Epoch 36/40, Loss: 0.02756
Epoch 37/40, Loss: 0.02610
Epoch 38/4

## Load trained model and save video with predicitons and inputs side-by-side

In [1]:



import os
import matplotlib.pyplot as plt
import numpy as np

# Inside this directory will be save a set of images comparing input and output (prediction)
os.makedirs("comparison_frames", exist_ok=True)

#model = UNet().to(device)
#model.load_state_dict(torch.load("unet_frame_predictor.pth", map_location=device))
model.eval()

with torch.no_grad():
    for i in range(len(video_dataset) - 1):  # avoid going out of bounds
        input_tensor, _ = video_dataset[i]
        true_next_tensor, _ = video_dataset[i + 1]

        # Prepare input and move to device
        inputs = input_tensor.unsqueeze(0).to(device)
        pred = model(inputs)

        # Convert tensors to numpy for plotting
        input_frame = input_tensor.numpy()[0]
        pred_frame = pred.cpu().numpy()[0, 0]
        true_next_frame = true_next_tensor.numpy()[0]

        # Create side-by-side-by-side image
        fig, axes = plt.subplots(1, 3, figsize=(9, 3))
        axes[0].imshow(input_frame, cmap='gray')
        axes[0].set_title("Input f(i)")
        axes[1].imshow(pred_frame, cmap='gray')
        axes[1].set_title("Prediction G(f(i))")
        axes[2].imshow(true_next_frame, cmap='gray')
        axes[2].set_title("True f(i+1)")
        for ax in axes:
            ax.axis('off')
        fig.tight_layout()

        # Save image
        plt.savefig(f"comparison_frames/frame_{i:03d}.png", bbox_inches='tight', pad_inches=0)
        plt.close()



import imageio.v2 as imageio
from glob import glob

# Collect all saved frame paths in order
frame_paths = sorted(glob("comparison_frames/frame_*.png"))

# Read images
frames = [imageio.imread(path) for path in frame_paths]

# Save as video
imageio.mimsave("comparison_video.mp4", frames, fps=10)
print("Output as video showcase is saved as: comparison_video.mp4")

# Note: some warnings appear, but they are not important for our case, video will show up aftewards
from IPython.display import Video
Video("comparison_video.mp4", embed=True)

NameError: name 'model' is not defined

## Task 2:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# Fix random seeds for reproducibility
np.random.seed(0)
torch.manual_seed(0)

# === Simulation Parameters ===
n_subjects = 3                 # Number of simulated subjects
n_samples_per_subject = 20    # Number of samples per subject
feature_dim = 10              # Arbitrary dimension of eye image features

# Simulate subject-specific corrections (e.g., angle kappa per person)
subject_corrections = {
    0: np.array([0.1, -0.05, 0.05]),
    1: np.array([-0.08, 0.03, 0.02]),
    2: np.array([0.0, 0.1, -0.1])
}

# === Generate synthetic data ===
features_all, gaze_base_all, gaze_true_all, subject_id_all = [], [], [], []

# For each subject, generate features, base gaze, and corrected true gaze
for sid, correction in subject_corrections.items():
    features = np.random.randn(n_samples_per_subject, feature_dim).astype(np.float32)
    gaze_base = np.random.randn(n_samples_per_subject, 3).astype(np.float32)
    gaze_base /= np.linalg.norm(gaze_base, axis=1, keepdims=True)      # Normalize to unit vector
    gaze_true = gaze_base + correction                                 # Apply subject correction
    gaze_true /= np.linalg.norm(gaze_true, axis=1, keepdims=True)      # Normalize again

    # Collect data
    features_all.append(features)
    gaze_base_all.append(gaze_base)
    gaze_true_all.append(gaze_true)
    subject_id_all.extend([sid] * n_samples_per_subject)

# Convert data to PyTorch tensors
features_all = torch.tensor(np.vstack(features_all))
gaze_base_all = torch.tensor(np.vstack(gaze_base_all))
gaze_true_all = torch.tensor(np.vstack(gaze_true_all))
subject_id_all = torch.tensor(subject_id_all)

# === Define model with subject embeddings ===
class CorrectionAgentMultiSubject(nn.Module):
    def __init__(self, feature_dim, n_subjects):
        super().__init__()
        self.subject_embedding = nn.Embedding(n_subjects, 8)  # 8D embedding per subject
        self.net = nn.Sequential(
            nn.Linear(feature_dim + 3 + 8, 64),  # input: features + base gaze + embedding
            nn.ReLU(),
            nn.Linear(64, 3)                     # output: correction vector
        )

    def forward(self, features, base_gaze, subject_ids):
        subject_emb = self.subject_embedding(subject_ids)
        x = torch.cat([features, base_gaze, subject_emb], dim=1)
        delta = self.net(x)
        return delta / torch.norm(delta, dim=1, keepdim=True)  # Normalize correction

# === Train the model ===
agent = CorrectionAgentMultiSubject(feature_dim, n_subjects)
optimizer = optim.Adam(agent.parameters(), lr=0.01)

losses = []
for epoch in range(100):
    optimizer.zero_grad()

    # Predict correction vector using the model
    delta = agent(features_all, gaze_base_all, subject_id_all)

    # Apply correction to base gaze and normalize
    corrected = gaze_base_all + delta
    corrected = corrected / corrected.norm(dim=1, keepdim=True)

    # Compute angular error between corrected and true gaze vectors
    dot_product = (corrected * gaze_true_all).sum(dim=1)
    angular_error = torch.acos(torch.clamp(dot_product, -1.0, 1.0))
    loss = angular_error.mean()

    # Backpropagation and optimization
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

# === Plot training progress ===
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Mean Angular Error (rad)")
plt.title("Training with Subject Embedding")
plt.grid(True)
plt.tight_layout()
plt.savefig("training_subject_embedding.png")
